In [1]:
import torch
from datasets import load_dataset
from transformers import Trainer, AutoModelForSequenceClassification, AutoTokenizer, AutoConfig, TrainingArguments
import os
import random

In [2]:
data_root_path = "../data"
model_root_path = "../models"
data_path = os.path.join(data_root_path, "zh-ru-trip-train-data-5k-no-id.csv")
model_path = os.path.join(model_root_path, "xlm-roberta-base")
model_path

'../models/xlm-roberta-base'

In [3]:
dataset = load_dataset("csv", data_files=data_path, split="train")
dataset

Dataset({
    features: ['queries', 'positive', 'negative'],
    num_rows: 5000
})

In [4]:
dataset = dataset.train_test_split(test_size=0.2)
# train_dataset = dataset["train"]
# train_dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['queries', 'positive', 'negative'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['queries', 'positive', 'negative'],
        num_rows: 1000
    })
})

In [5]:
def split_examples(batch):
    queries = []
    passages = []
    labels = []
    for label in ["positive", "negative"]:
        for (query, passage) in zip(batch["queries"], batch[label]):
            queries.append(query)
            passages.append(passage)
            labels.append(int(label == "positive"))
    return {"query": queries, "passage": passages, "label": labels}

dataset = dataset.map(split_examples, batched=True, remove_columns=["queries", "positive", "negative"])
dataset
# print(train_dataset)
# print(train_dataset[:2])

Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['query', 'passage', 'label'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['query', 'passage', 'label'],
        num_rows: 2000
    })
})

In [6]:
train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [7]:
# 打乱数据
# import random

# # 获取示例的索引列表
# indices = list(range(len(train_dataset))

# # 随机打乱示例的顺序
# random.shuffle(indices)

# # 使用打乱后的索引列表重新排列 train_dataset
# train_dataset = train_dataset.select(indices)

In [9]:
train_indices = list(range(len(train_dataset)))
random.shuffle(train_indices)
train_dataset = train_dataset.select(train_indices)

test_indices = list(range(len(test_dataset)))
random.shuffle(test_indices)
test_dataset = test_dataset.select(test_indices)

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_path)
config = AutoConfig.from_pretrained(model_path)
config.num_labels = 1
config.problem_type = "multi_label_classification"
model = AutoModelForSequenceClassification.from_pretrained(model_path, config=config)

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at ../models/xlm-roberta-base and are newly initialized: ['classifier.out_proj.weight', 'classifier.out_proj.bias', 'classifier.dense.weight', 'classifier.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [16]:
tokenizer

XLMRobertaTokenizerFast(name_or_path='../models/xlm-roberta-base', vocab_size=250002, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	250001: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=False, special=True),
}

In [11]:
def tokenize(batch):
    tokenized = tokenizer(
        batch["query"],
        batch["passage"],
        padding=True,
        truncation=True,
        max_length=128,
    )
    tokenized["labels"] = [[float(label)] for label in batch["label"]]
    return tokenized

train_tokenized_dataset = train_dataset.map(tokenize, batched=True, remove_columns=["query", "passage", "label"])
test_tokenized_dataset = test_dataset.map(tokenize, batched=True, remove_columns=["query", "passage", "label"])

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [15]:
print(train_tokenized_dataset[3])

{'input_ids': [0, 36097, 44507, 145219, 354, 1173, 6711, 42930, 2, 2, 17597, 1730, 222, 12635, 44627, 1041, 20, 1097, 44627, 2233, 4427, 551, 4, 2032, 2582, 2233, 49, 729, 5046, 8165, 4, 135, 164213, 18372, 1757, 56242, 419, 4, 38863, 91957, 49, 9880, 130, 15052, 2687, 4, 147332, 4, 49, 100192, 8165, 255, 33690, 6, 140073, 183, 75674, 8865, 78004, 2429, 328, 1200, 888, 245, 310, 5, 130993, 39331, 49, 64110, 187114, 1214, 12, 53302, 1394, 13587, 20, 85477, 41747, 3281, 63059, 21967, 35, 37235, 29034, 3988, 227, 4, 135, 226551, 27965, 1097, 6, 30462, 512, 49517, 187114, 4, 252, 62936, 152217, 312, 20, 85477, 29980, 3429, 41228, 312, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

In [17]:
train_tokenized_dataset.set_format("torch")
test_tokenized_dataset.set_format("torch")

In [18]:
training_args = TrainingArguments(output_dir='./XLM-R_output',
                                  fp16=True,
                                  half_precision_backend="auto",
                                  per_device_train_batch_size=32,
                                  num_train_epochs=3,
                                  save_strategy="epoch",
                                  save_total_limit=3,
                                  logging_steps=50,
                                  warmup_steps=100)

In [19]:
trainer = Trainer(model=model,
                  args=training_args,
                  train_dataset=train_tokenized_dataset,
                  tokenizer=tokenizer)

In [20]:
trainer.train()

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
50,0.694400
100,0.658600
150,0.573800
200,0.565800
250,0.493900
300,0.416200
350,0.487700
400,0.493000
450,0.400400
500,0.336500


TrainOutput(global_step=750, training_loss=0.4278592694600423, metrics={'train_runtime': 154.1317, 'train_samples_per_second': 155.711, 'train_steps_per_second': 4.866, 'total_flos': 1578652157952000.0, 'train_loss': 0.4278592694600423, 'epoch': 3.0})

In [21]:
trainer.train()

Step,Training Loss
50,0.214900
100,0.277700
150,0.330900
200,0.329300
250,0.269600
300,0.201000
350,0.194700
400,0.245400
450,0.311700
500,0.234100


TrainOutput(global_step=750, training_loss=0.23260111300150554, metrics={'train_runtime': 160.0845, 'train_samples_per_second': 149.921, 'train_steps_per_second': 4.685, 'total_flos': 1578652157952000.0, 'train_loss': 0.23260111300150554, 'epoch': 3.0})